# Run code-server on Kaggle

This notebook installs code-server and cloudflared, opens the `/kaggle/working` workspace, and creates a Cloudflare Quick Tunnel for browser access. Internet access must be enabled in Kaggle.

> **Security:** The password and public URL are printed when the services start. Do not share this notebook or its outputs while the services are running. Run `stop_services()` in the last cell when finished.


## 1. Install the required tools

Run this cell first. It installs code-server and cloudflared, then checks their versions. Kaggle Internet access is required.

In [ ]:
import os
import re
import secrets
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen

CODE_SERVER_PORT = 8080
WORKSPACE = Path("/kaggle/working")
CONFIG_DIR = Path.home() / ".config" / "code-server"
CONFIG_FILE = CONFIG_DIR / "config.yaml"
CODE_LOG = WORKSPACE / "code-server.log"
TUNNEL_LOG = WORKSPACE / "cloudflared.log"
CLOUDFLARED = Path("/usr/local/bin/cloudflared")
CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64"
)

PASSWORD = None
public_url = None
code_process = None
tunnel_process = None
code_log_file = None
tunnel_log_file = None


def install_code_server():
    """Install code-server using its official installer."""
    print("Installing code-server...")
    subprocess.run(
        [
            "bash", "-o", "pipefail", "-c",
            "curl -fsSL https://code-server.dev/install.sh | "
            "sh -s -- --method=standalone",
        ],
        check=True,
    )

    local_bin = str(Path.home() / ".local" / "bin")
    os.environ["PATH"] = f"{local_bin}:{os.environ['PATH']}"
    version = subprocess.run(
        ["code-server", "--version"],
        check=True,
        capture_output=True,
        text=True,
    )
    print("code-server:", version.stdout.strip())


def install_cloudflared():
    """Install cloudflared if it is missing."""
    if not CLOUDFLARED.exists():
        print("Downloading cloudflared...")
        CLOUDFLARED.parent.mkdir(parents=True, exist_ok=True)
        with urlopen(CLOUDFLARED_URL, timeout=30) as response:
            CLOUDFLARED.write_bytes(response.read())
        CLOUDFLARED.chmod(0o755)

    version = subprocess.run(
        [str(CLOUDFLARED), "--version"],
        check=True,
        capture_output=True,
        text=True,
    )
    print("cloudflared:", version.stdout.strip())


def is_code_server_healthy():
    """Return whether code-server responds on localhost."""
    health_url = f"http://127.0.0.1:{CODE_SERVER_PORT}/healthz"
    try:
        with urlopen(health_url, timeout=2) as response:
            return response.status == 200
    except Exception:
        return False


install_code_server()
install_cloudflared()


## 2. Start code-server

This cell creates a random password, writes a private configuration file, and starts code-server on localhost. Copy the password from the output and keep it private.

In [ ]:
def start_code_server():
    """Write the configuration and start code-server in the Kaggle workspace."""
    global PASSWORD, code_process, code_log_file

    if code_process is not None and code_process.poll() is None:
        print("code-server is already running.")
        return
    if is_code_server_healthy():
        raise RuntimeError(
            "A code-server process is already running outside this kernel. "
            "Restart the Kaggle runtime before starting another one."
        )

    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    PASSWORD = secrets.token_urlsafe(24)
    CONFIG_FILE.write_text(
        f"bind-addr: 127.0.0.1:{CODE_SERVER_PORT}\n"
        "auth: password\n"
        f"password: {PASSWORD}\n"
        "cert: false\n",
        encoding="utf-8",
    )
    CONFIG_FILE.chmod(0o600)

    code_log_file = CODE_LOG.open("w", encoding="utf-8")
    code_process = subprocess.Popen(
        [
            "code-server",
            "--config", str(CONFIG_FILE),
            "--disable-telemetry",
            str(WORKSPACE),
        ],
        stdout=code_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    for _ in range(30):
        if is_code_server_healthy():
            print("code-server is ready.")
            print("Workspace:", WORKSPACE)
            print("Password (keep private):", PASSWORD)
            return
        if code_process.poll() is not None:
            break
        time.sleep(1)

    log_tail = CODE_LOG.read_text(encoding="utf-8", errors="ignore")[-5000:]
    raise RuntimeError(f"code-server failed to start. Log:\n{log_tail}")


start_code_server()


## 3. Create a Cloudflare Tunnel

This cell exposes the local code-server through a temporary public URL. Open the URL and sign in with the password from the previous cell. Treat both as private.

In [ ]:
def start_tunnel():
    """Create a Quick Tunnel to code-server and print its public URL."""
    global public_url, tunnel_process, tunnel_log_file

    if tunnel_process is not None and tunnel_process.poll() is None:
        print("Cloudflare Tunnel is already running:", public_url or "URL pending")
        return
    if not is_code_server_healthy():
        raise RuntimeError("Start code-server before opening the tunnel.")

    tunnel_log_file = TUNNEL_LOG.open("w", encoding="utf-8")
    tunnel_process = subprocess.Popen(
        [
            str(CLOUDFLARED), "tunnel",
            "--url", f"http://127.0.0.1:{CODE_SERVER_PORT}",
            "--no-autoupdate",
        ],
        stdout=tunnel_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    for _ in range(60):
        if tunnel_process.poll() is not None:
            break
        log_text = TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            print("code-server is ready.")
            print("URL:", public_url)
            print("Use the password from the previous cell to sign in.")
            return
        time.sleep(1)

    log_tail = TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")[-5000:]
    raise RuntimeError(f"Could not get a Cloudflare URL. Log:\n{log_tail}")


start_tunnel()


## 4. Check service status

This cell checks whether code-server responds and whether the Cloudflare Tunnel process is running.

In [ ]:
print("code-server:", "RUNNING" if is_code_server_healthy() else "STOPPED")
print(
    "Cloudflare Tunnel:",
    "RUNNING" if tunnel_process is not None and tunnel_process.poll() is None else "STOPPED",
)


## 5. Inspect service logs

Use this cell to view the most recent code-server and cloudflared log lines when diagnosing a problem.

In [ ]:
def show_logs(lines=80):
    """Print the last lines from each service log."""
    for log_path in (CODE_LOG, TUNNEL_LOG):
        print(f"\n--- {log_path.name} ---")
        if log_path.exists():
            log_lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
            print("\n".join(log_lines[-lines:]))
        else:
            print("Log file does not exist.")


show_logs()


## 6. Stop the services

When finished, uncomment and run `stop_services()` below to stop the tunnel and code-server.

In [ ]:
def stop_services():
    """Stop the tunnel and code-server when they are no longer needed."""
    global tunnel_process, code_process

    for name, process in (
        ("Cloudflare Tunnel", tunnel_process),
        ("code-server", code_process),
    ):
        if process is None or process.poll() is not None:
            print(f"{name}: already stopped.")
            continue

        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)
        print(f"{name}: stopped.")

    for log_file in (tunnel_log_file, code_log_file):
        if log_file is not None and not log_file.closed:
            log_file.close()


# Run this when you want to shut down the services:
# stop_services()
